# RAG Retrieval Augmented Generation (Groq)
## Carga de documentos PDF al Prompt Aumentado para el LLM

Sistema **RAG** completo usando **Groq** como LLM (gratuito, ultra-rápido) y **Sentence Transformers** para embeddings locales (sin GPU, corre en Colab).

### Pipeline del sistema:

```
📄 Documentos PDF
       │
       ▼
  ✂️  PASO 1 — Carga de documentos
       │
       ▼
  🧩  PASO 2 — División en fragmentos  (Chunking)
       │
       ▼
  🔢  PASO 3 — Embeddings locales  ← sentence-transformers (CPU, Colab)
       │
       ▼
  🗄️  PASO 4 — Almacenamiento en Base Vectorial (ChromaDB)
       │
       ▼
  ❓  Consulta del usuario
       │
       ▼
  🔍  PASO 5 — Búsqueda vectorial → Fragmentos relevantes
       │
       ▼
  📝  PASO 6 — Construcción del Prompt Aumentado
       │
       ▼
  🤖  PASO 7 — LLM genera respuesta  ← Groq API (gratuita)
       │
       ▼
  📊  PASO 8 — Evaluación RAGAs
```

**Tecnologías:** LangChain · ChromaDB · Sentence Transformers · Groq (llama-3.3-70b)


## ⚙️ Configuración del Entorno

### ¿Por qué Groq?
- **Gratuito**: tier gratuito generoso (14.400 solicitudes/día, 500.000 tokens/minuto)
- **Ultra-rápido**: inferencia en hardware especializado (LPU), mucho más rápido que GPU
- **Sin instalación local**: solo necesitas una API key

### Obtener API Key de Groq (gratis)
1. Ve a **https://console.groq.com**
2. Crea una cuenta gratuita
3. En **API Keys** → **Create API Key**
4. Copia la key y pégala en la celda de abajo

### Modelos disponibles en Groq (tier gratuito)
| Modelo | Descripción |
|---|---|
| `llama-3.3-70b-versatile` | ✅ Recomendado — muy capaz, rápido |
| `llama-3.1-8b-instant` | Ultra-rápido, menor calidad |
| `mixtral-8x7b-32768` | Contexto largo (32k tokens) |
| `gemma2-9b-it` | Alternativa ligera de Google |


In [37]:
%pip install --upgrade \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-chroma \
    langchain-huggingface \
    langchain-groq \
    ragas \
    sentence-transformers \
    chromadb \
    pypdf

In [38]:
# CELDA 1 — Parche para el bug de VertexAI en ragas (ejecutar primero)
import sys
import unittest.mock as mock

sys.modules['langchain_community.chat_models.vertexai'] = mock.MagicMock()
sys.modules['langchain_community.llms.vertexai'] = mock.MagicMock()

print("[OK] Parche aplicado")

[OK] Parche aplicado


In [39]:
import importlib.metadata

paquetes = [
    "langchain-core",
    "langchain-community",
    "langchain-huggingface",
    "langchain-groq",
    "langchain-chroma",
    "ragas",
    "sentence-transformers",
    "chromadb",
]

for paquete in paquetes:
    try:
        version = importlib.metadata.version(paquete)
        print(f"  {paquete:<30} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {paquete:<30} ❌ no instalado")

  langchain-core                 1.6.3
  langchain-community            0.4.2
  langchain-huggingface          1.2.2
  langchain-groq                 1.1.3
  langchain-chroma               1.1.0
  ragas                          0.4.3
  sentence-transformers          6.0.1
  chromadb                       1.5.9


In [40]:
import os
from google.colab import userdata

# Opción 1: desde los Secrets de Colab (recomendado)
# En Colab: ícono 🔑 (panel izquierdo) → añadir secreto GROQ_API_KEY
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print(f"[OK] API Key cargada desde Secrets de Colab (...{GROQ_API_KEY[-6:]})")
except Exception:
    # Opción 2: pegar la key directamente (menos seguro)
    GROQ_API_KEY = "gsk_PEGA_TU_KEY_AQUI"
    print("[!]  API Key cargada manualmente — considera usar Secrets de Colab")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY


[OK] API Key cargada desde Secrets de Colab (...5lnqO4)


## 🧠 Paso 0 — Búsqueda Semántica

Antes de implementar el pipeline, entendamos por qué la búsqueda con **embeddings** supera a la búsqueda por palabras clave.

| | Keyword Search | Búsqueda Semántica |
|---|---|---|
| Consulta | *"ayuda de dinero por notas excelentes"* | *"ayuda de dinero por notas excelentes"* |
| Documento | *"incentivo económico por promedio superior a 4.5"* | *"incentivo económico por promedio superior a 4.5"* |
| ¿Encuentra? | ❌ No (no comparte palabras) | ✅ Sí (mismo significado) |

Los **embeddings** se generan localmente con **Sentence Transformers** — sin costo, sin API key, corren en CPU en Colab.


In [42]:
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Modelo de embeddings local — corre en CPU, se descarga ~120 MB la primera vez
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print(f"[OK] Embeddings locales cargados: {EMBEDDING_MODEL}")

# Corpus de ejemplo
reglamento = [
    "Los alumnos con promedio superior a 4.5 reciben un incentivo económico.",
    "El abandono de los estudios sin previo aviso causa sanción administrativa.",
    "Se pueden pedir exámenes extraordinarios si hay una causa médica comprobada.",
    "La universidad ofrece apoyo financiero para proyectos de investigación.",
]

corpus_embeddings = embeddings.embed_documents(reglamento)
query = "ayuda de dinero por notas excelentes"
query_embedding = embeddings.embed_query(query)

scores = cosine_similarity([query_embedding], corpus_embeddings)
indices_ordenados = np.argsort(scores[0])[::-1]

print(f"\nConsulta: \"{query}\"\n")
for idx in indices_ordenados:
    barra = "█" * int(scores[0][idx] * 20)
    print(f"  {scores[0][idx]:.4f} {barra}")
    print(f"  {reglamento[idx]}\n")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[OK] Embeddings locales cargados: paraphrase-multilingual-MiniLM-L12-v2

Consulta: "ayuda de dinero por notas excelentes"

  0.5063 ██████████
  La universidad ofrece apoyo financiero para proyectos de investigación.

  0.3901 ███████
  Los alumnos con promedio superior a 4.5 reciben un incentivo económico.

  0.2653 █████
  Se pueden pedir exámenes extraordinarios si hay una causa médica comprobada.

  0.1186 ██
  El abandono de los estudios sin previo aviso causa sanción administrativa.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 📄 Paso 1 — Carga de Documentos PDF

LangChain proporciona **loaders** para cargar distintos tipos de documentos. Para PDFs usamos `PyPDFLoader`, que extrae el texto de cada página y lo envuelve en un objeto `Document` con:
- `page_content`: texto de la página
- `metadata`: información del documento (fuente, número de página, etc.)

Cargaremos los documentos PDF del directorio `pdfs/`.


In [43]:
import os
from langchain_community.document_loaders import PyPDFLoader

pdf_dir = "pdfs"
pdf_files = sorted([f for f in os.listdir(pdf_dir) if f.endswith(".pdf")])

print(f"PDFs disponibles en \"{pdf_dir}/\" ({len(pdf_files)} archivos):")
for i, f in enumerate(pdf_files, 1):
    size_kb = os.path.getsize(os.path.join(pdf_dir, f)) // 1024
    print(f"  [{i}] {f}  ({size_kb} KB)")


PDFs disponibles en "pdfs/" (2 archivos):
  [1] 1-11015-Pre-Cálculo.pdf  (429 KB)
  [2] Acuerdo-No.-01-de-2025-Reglamento-Academico-Institucional.pdf  (620 KB)


In [44]:
documents = []

for pdf_file in pdf_files:
    path = os.path.join(pdf_dir, pdf_file)
    loader = PyPDFLoader(path)
    pages = loader.load()
    documents.extend(pages)
    print(f"  {pdf_file}: {len(pages)} páginas cargadas")

print(f"\nTotal de páginas cargadas: {len(documents)}")


  1-11015-Pre-Cálculo.pdf: 6 páginas cargadas
  Acuerdo-No.-01-de-2025-Reglamento-Academico-Institucional.pdf: 64 páginas cargadas

Total de páginas cargadas: 70


In [45]:
doc_ejemplo = documents[0]

print("Estructura de un objeto Document:")
print(f"  Tipo: {type(doc_ejemplo)}")
print()
print("Metadatos (metadata):")
for k, v in doc_ejemplo.metadata.items():
    print(f"  {k}: {v}")
print()
print(f"Primeros 500 caracteres de page_content:")
print("-" * 50)
print(doc_ejemplo.page_content[:500])


Estructura de un objeto Document:
  Tipo: <class 'langchain_core.documents.base.Document'>

Metadatos (metadata):
  producer: Microsoft® Word para Microsoft 365
  creator: Microsoft® Word para Microsoft 365
  creationdate: 2026-02-09T12:25:47-05:00
  author: Optimización y Mejoramiento
  moddate: 2026-02-09T12:25:47-05:00
  source: pdfs/1-11015-Pre-Cálculo.pdf
  total_pages: 6
  page: 0
  page_label: 1

Primeros 500 caracteres de page_content:
--------------------------------------------------
FR-PD-G-501 
Programa de Asignatura 
 Versión 6.0 Página 1 de 6 
 
Optimización y Mejoramiento, actualizado 06/08/2024 
Facultad/ Escuela/ Instituto: Facultad de Matemáticas e Ingenierías 
Programa académico: Matemáticas - Ingeniería de Sistemas - Ingeniería Industrial 
Nivel de formación: ☒ Pregrado             ☐ Posgrado 
Modalidad del programa: ☒ Presencial           ☐ Virtual          ☐ Híbrido 
 
Nombre de la 
asignatura: Pre-Cálculo Código: 11015 Semestre: I 
Modalidad de 
la asignatura: ☒


## ✂️ Paso 2 — División en Fragmentos (Chunking)

Los modelos de embeddings tienen un **límite de tokens** de entrada. Dividir los documentos en fragmentos pequeños permite:

1. **Precisión**: recuperar solo la información relevante (no páginas enteras)
2. **Compatibilidad**: respetar los límites del modelo de embeddings
3. **Eficiencia**: vectores más pequeños → búsqueda más rápida

| Parámetro | Valor | Descripción |
|---|---|---|
| `chunk_size` | 500 | Máximo de caracteres por fragmento |
| `chunk_overlap` | 50 | Caracteres compartidos entre fragmentos consecutivos |


In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(documents)

print(f"Documentos originales (páginas): {len(documents)}")
print(f"Fragmentos generados:            {len(chunks)}")
print(f"Factor de expansión:             {len(chunks)/len(documents):.1f}x")
print()
print("Ejemplo — Fragmento #20:")
print(f"  Fuente: {chunks[20].metadata.get('source', '?')}")
print(f"  Página: {chunks[20].metadata.get('page', '?')}")
print(f"  Longitud: {len(chunks[20].page_content)} caracteres")
print(f"  Contenido: {chunks[20].page_content}")


Documentos originales (páginas): 70
Fragmentos generados:            304
Factor de expansión:             4.3x

Ejemplo — Fragmento #20:
  Fuente: pdfs/1-11015-Pre-Cálculo.pdf
  Página: 4
  Longitud: 495 caracteres
  Contenido: • Utiliza vocabulario y términos específicos del Pre-Cálculo 
en su lengua materna o en un segundo idioma. 
• Determina validez o invalidez de procesos matemáticos 
desarrollados en Pre-Cálculo. 
• Aplica conceptos matemáticos básicos para resolver 
problemas de Pre-Cálculo. 
Temáticas: Estrategias de evaluación: Medios educativos: 
• Definición de la 
función 
exponencial y 
logaritmo. 
• Propiedades de los 
logaritmos. 
• Ecuaciones que 
involucran 
logaritmos y 
exponenciales. 
• Modelos


## 🔢 Paso 3 — Generación de Embeddings (local, CPU)

**Modelo:** `paraphrase-multilingual-MiniLM-L12-v2`
- Dimensión: **384** componentes por fragmento
- Multilingüe — soporta español perfectamente
- Corre en **CPU** en Colab, sin necesidad de GPU
- Se descarga automáticamente (~120 MB) y queda en caché


In [47]:
# El modelo ya fue cargado en el Paso 0 — lo reutilizamos
embeddings_model = embeddings

sample_text = chunks[20].page_content
sample_vector = embeddings_model.embed_query(sample_text)

print("Texto de muestra:")
print(f"  {sample_text[:120]}...")
print()
print("Embedding generado:")
print(f"  Dimensiones del vector: {len(sample_vector)}")
print(f"  Rango de valores:       [{min(sample_vector):.4f},  {max(sample_vector):.4f}]")
print(f"  Primeros 8 valores:     {[round(v, 4) for v in sample_vector[:8]]}")


Texto de muestra:
  • Utiliza vocabulario y términos específicos del Pre-Cálculo 
en su lengua materna o en un segundo idioma. 
• Determina ...

Embedding generado:
  Dimensiones del vector: 384
  Rango de valores:       [-0.1279,  0.1361]
  Primeros 8 valores:     [-0.0728, 0.0018, 0.0272, -0.0162, -0.1191, -0.0173, -0.0417, 0.0981]


## 🗄️ Paso 4 — Almacenamiento en Base Vectorial (ChromaDB)

**ChromaDB** almacena los vectores generados con Sentence Transformers y permite búsqueda por similitud coseno. Todo corre localmente en Colab, sin servicios externos.


In [49]:
from langchain_chroma import Chroma
import shutil

PERSIST_DIR = "./chroma"

if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)
    print(f"Base de datos anterior eliminada: {PERSIST_DIR}")

print(f"Indexando {len(chunks)} fragmentos en ChromaDB...")
print("(Embeddings generados localmente con Sentence Transformers)")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    persist_directory=PERSIST_DIR,
    collection_name="mis_programas",
    collection_metadata={"hnsw:space": "cosine"}
)

total = vector_store._collection.count()
print()
print("[OK] Base vectorial creada exitosamente!")
print(f"  Ubicación:             {PERSIST_DIR}/")
print(f"  Fragmentos indexados:  {total}")
print(f"  Modelo embeddings:     {EMBEDDING_MODEL} (local, CPU)")
print(f"  Dimensión de vectores: 384")


Indexando 304 fragmentos en ChromaDB...
(Embeddings generados localmente con Sentence Transformers)

[OK] Base vectorial creada exitosamente!
  Ubicación:             ./chroma/
  Fragmentos indexados:  304
  Modelo embeddings:     paraphrase-multilingual-MiniLM-L12-v2 (local, CPU)
  Dimensión de vectores: 384


In [50]:
# Cargar base vectorial existente sin re-procesar (útil para reanudar)
# vector_store = Chroma(
#     persist_directory=PERSIST_DIR,
#     embedding_function=embeddings_model
# )
# print(f"Base vectorial cargada: {vector_store._collection.count()} fragmentos")

coleccion = vector_store._collection
print("Información de la colección ChromaDB:")
print(f"  Nombre:              {coleccion.name}")
print(f"  Total documentos:    {coleccion.count()}")
print(f"  Metadata:            {coleccion.metadata}")


Información de la colección ChromaDB:
  Nombre:              mis_programas
  Total documentos:    304
  Metadata:            {'hnsw:space': 'cosine'}


## 🔍 Paso 5 — Recuperación Vectorial (Retrieval)

El **retriever** recibe una consulta en lenguaje natural y devuelve los fragmentos más relevantes usando los embeddings locales para buscar en ChromaDB.


In [51]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

pregunta = "cuales son las distinciones academicas"

print(f"Consulta: \"{pregunta}\"")
print(f"Recuperando los 5 fragmentos más relevantes...\n")

documentos_recuperados = retriever.invoke(pregunta)

print(f"Fragmentos recuperados: {len(documentos_recuperados)}")
print("=" * 60)
for i, doc in enumerate(documentos_recuperados, 1):
    fuente = os.path.basename(doc.metadata.get("source", "desconocido"))
    pagina = doc.metadata.get("page", "?")
    print(f"\n[{i}] {fuente} — Pág. {pagina} ({len(doc.page_content)} chars)")
    print(f"    {doc.page_content[:250]}...")


Consulta: "cuales son las distinciones academicas"
Recuperando los 5 fragmentos más relevantes...

Fragmentos recuperados: 10

[1] Acuerdo-No.-01-de-2025-Reglamento-Academico-Institucional.pdf — Pág. 40 (467 chars)
    académico objeto de distinción. 
 
PARÁGRAFO 2: En el caso de estudiantes que participen en 
programas de intercambio académico y homologuen asignaturas 
cursadas en instituciones nacionales o internacionales,  podrán ser 
considerados para optar por...

[2] Acuerdo-No.-01-de-2025-Reglamento-Academico-Institucional.pdf — Pág. 23 (135 chars)
    aprendizaje establecidos por el programa académico. Los medios en 
los que se realice la evaluación académica deben  corresponder a las...

[3] Acuerdo-No.-01-de-2025-Reglamento-Academico-Institucional.pdf — Pág. 1 (386 chars)
    especialización, maestría y doctorado. 
  
 Se entiende por programas académicos de pregrado aquellos que 
conducen a la obtención de un título en los niveles Técnico 
Profesional, Tecnológico o Profesio

## 📝 Paso 6 — Construcción del Prompt Aumentado

Este es el **corazón del RAG**: enriquecer la pregunta con el contexto recuperado antes de enviarlo al LLM.

```
SISTEMA: Instrucciones de comportamiento
─────────────────────────────────────
CONTEXTO: [Fragmento 1 - Pág. X]
          texto del fragmento 1...
─────────────────────────────────────
PREGUNTA: ¿Cuáles son los temas del curso?
─────────────────────────────────────
RESPUESTA: (generada por Groq)
```


In [52]:
from langchain_core.prompts import ChatPromptTemplate
import os

PROMPT_TEMPLATE = '''Eres un asistente legal experto en reglamentos y normas.
Responde la pregunta usando ÚNICAMENTE la información del contexto proporcionado. Al final de cada parte de la respuesta, incluye entre paréntesis la fuente y la página del fragmento de donde proviene la información, por ejemplo: (Fuente: NombreDocumento.pdf, Pág. X).
Si la respuesta no está en el contexto, indica exactamente: "No encontré información sobre esto en la base de conocimientos."

Contexto recuperado del documento:
{context}

Pregunta del usuario: {question}

Respuesta:'''

prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

contexto = "\n\n---\n\n".join(
    f"[Fuente: {os.path.basename(doc.metadata.get('source', '?'))} — Pág. {doc.metadata.get('page', '?')}]\n{doc.page_content}"
    for i, doc in enumerate(documentos_recuperados)
)

prompt_aumentado = prompt_template.invoke({
    "context": contexto,
    "question": pregunta
})

print("Prompt aumentado construido correctamente.")
print(f"  Fragmentos en el contexto:       {len(documentos_recuperados)}")
print(f"  Caracteres totales del contexto: {len(contexto)}")
print(f"  Tokens aproximados del contexto: ~{len(contexto)//4}")

Prompt aumentado construido correctamente.
  Fragmentos en el contexto:       10
  Caracteres totales del contexto: 4647
  Tokens aproximados del contexto: ~1161


In [53]:
print("=" * 65)
print("ESTRUCTURA DEL PROMPT AUMENTADO (RAG)")
print("=" * 65)

print("\n[SISTEMA — Instrucciones para el LLM]")
print("  \"Eres un asistente académico especializado...\"")

print("\n[CONTEXTO — Fragmentos recuperados del vector store]")
for i, doc in enumerate(documentos_recuperados, 1):
    fuente = os.path.basename(doc.metadata.get("source", "?"))
    pagina = doc.metadata.get("page", "?")
    print(f"  [{i}] {fuente} Pág.{pagina} — {len(doc.page_content)} chars")
    print(f"       \"{doc.page_content[:80]}...\"")

print(f"\n[PREGUNTA DEL USUARIO]")
print(f"  \"{pregunta}\"")
print("\n[RESPUESTA]")
print("  (será generada por Groq en el siguiente paso)")
print("=" * 65)


ESTRUCTURA DEL PROMPT AUMENTADO (RAG)

[SISTEMA — Instrucciones para el LLM]
  "Eres un asistente académico especializado..."

[CONTEXTO — Fragmentos recuperados del vector store]
  [1] Acuerdo-No.-01-de-2025-Reglamento-Academico-Institucional.pdf Pág.40 — 467 chars
       "académico objeto de distinción. 
 
PARÁGRAFO 2: En el caso de estudiantes que pa..."
  [2] Acuerdo-No.-01-de-2025-Reglamento-Academico-Institucional.pdf Pág.23 — 135 chars
       "aprendizaje establecidos por el programa académico. Los medios en 
los que se re..."
  [3] Acuerdo-No.-01-de-2025-Reglamento-Academico-Institucional.pdf Pág.1 — 386 chars
       "especialización, maestría y doctorado. 
  
 Se entiende por programas académicos..."
  [4] Acuerdo-No.-01-de-2025-Reglamento-Academico-Institucional.pdf Pág.46 — 481 chars
       "del último periodo cursado. 
 
ARTÍCULO 105: La Institución puede otorgar, a las..."
  [5] Acuerdo-No.-01-de-2025-Reglamento-Academico-Institucional.pdf Pág.19 — 55 chars
       "académi

## 🤖 Paso 7 — Generación con el LLM (Groq)

El LLM recibe el **prompt aumentado** y genera una respuesta basada **únicamente** en el contexto recuperado.

**Motor:** Groq API — inferencia ultrarrápida sobre hardware LPU  
**Modelo:** `llama-3.3-70b-versatile` — potente, gratuito, excelente en español  
**Temperature:** `0.0` — respuestas deterministas, ideal para QA sobre documentos


In [54]:
from langchain_groq import ChatGroq

GROQ_MODEL = "openai/gpt-oss-120b"
# Alternativas gratuitas en Groq:
# "llama-3.1-8b-instant"   → más rápido, menor calidad
# "mixtral-8x7b-32768"     → contexto largo (32k tokens)
# "gemma2-9b-it"           → alternativa ligera

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.0,
    api_key=GROQ_API_KEY
)

print(f"[OK] LLM configurado: {GROQ_MODEL} (vía Groq API)")
print()
print(f"Enviando prompt aumentado a Groq...")
print(f"Pregunta: \"{pregunta}\"")
print()

respuesta = llm.invoke(prompt_aumentado)
texto_respuesta = respuesta.content

print("=" * 60)
print("RESPUESTA DEL LLM (basada en el contexto RAG):")
print("=" * 60)
print(texto_respuesta)


[OK] LLM configurado: openai/gpt-oss-120b (vía Groq API)

Enviando prompt aumentado a Groq...
Pregunta: "cuales son las distinciones academicas"

RESPUESTA DEL LLM (basada en el contexto RAG):
Según el Reglamento Académico Institucional, la institución otorga a los estudiantes las siguientes distinciones académicas:

1. **Mención de Honor**  
2. **Matrícula de Honor**  
3. **Grado de Honor**  
4. **Grado Meritorio**  

(Fuente: Acuerdo No. 01‑2025 Reglamento Académico Institucional, pág. 39)


In [55]:
# Comparación: RAG vs LLM sin contexto
pregunta_test = "¿Qué estudiantes están matriculados en el curso?"

# Con RAG
docs_test = retriever.invoke(pregunta_test)
contexto_test = "\n\n---\n\n".join(
    f"[Fragmento {i+1}]\n{d.page_content}" for i, d in enumerate(docs_test)
)
prompt_con_rag = prompt_template.invoke({"context": contexto_test, "question": pregunta_test})
texto_con_rag = llm.invoke(prompt_con_rag).content

# Sin RAG
from langchain_core.messages import HumanMessage
texto_sin_rag = llm.invoke([HumanMessage(content=pregunta_test)]).content

print(f"Pregunta: \"{pregunta_test}\"")
print()
print("[CON RAG — basado en el documento]")
print(texto_con_rag)
print()
print("[SIN RAG — solo conocimiento del LLM]")
print(texto_sin_rag)


Pregunta: "¿Qué estudiantes están matriculados en el curso?"

[CON RAG — basado en el documento]
Según el reglamento institucional, están matriculados en un curso los siguientes tipos de estudiantes:

1. **Estudiantes regulares** – Aquellos que se han matriculado en cualquiera de los programas académicos que conducen a un título académico formal y que cumplen con los requisitos exigidos por la normativa legal, los Estatutos y los Reglamentos de la Institución. (Fuente: Fragmento 2, Artículo 39)

2. **Estudiantes de extensión** – Aquellos que cursan asignaturas de programas académicos sin pertenecer formalmente a ellos. (Fuente: Fragmento 2, Artículo 40)

3. **Estudiantes que formalizan su matrícula mediante acta** – La inscripción se formaliza mediante un acta suscrita por el Decano y el Coordinador Académico. (Fuente: Fragmento 1)

4. **Estudiantes que cumplen con los requisitos pecuniarios y administrativos** – La calidad de estudiante matriculado entra en vigor a partir del pago de 

## 🔄 Pipeline RAG Completo

Función reutilizable que encapsula el flujo completo: **consulta → recuperación (local) → prompt aumentado → respuesta (Groq)**.


In [56]:
import os
def rag_pipeline(pregunta: str, k: int = 5, verbose: bool = False) -> dict:
    '''
    Pipeline RAG de extremo a extremo.

    Embeddings: Sentence Transformers (local, CPU)
    Vector store: ChromaDB (local)
    LLM: Groq API (llama-3.3-70b, gratuito)
    '''
    retriever_k = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    docs = retriever_k.invoke(pregunta)

    contexto = "\n\n---\n\n".join(
        f"[Fuente: {os.path.basename(d.metadata.get('source', '?'))} — Pág. {d.metadata.get('page','?')}]\n{d.page_content}"
        for i, d in enumerate(docs)
    )

    prompt = prompt_template.invoke({"context": contexto, "question": pregunta})
    texto = llm.invoke(prompt).content

    if verbose:
        print(f"Fragmentos recuperados: {len(docs)}")
        for i, d in enumerate(docs, 1):
            fuente = os.path.basename(d.metadata.get("source", "?"))
            pagina = d.metadata.get("page", "?")
            print(f"  [{i}] {fuente} Pág.{pagina}: {d.page_content[:80]}...")

    return {
        "pregunta": pregunta,
        "fragmentos": docs,
        "contexto": contexto,
        "respuesta": texto,
        "tokens_contexto_aprox": len(contexto) // 4
    }

print("Función rag_pipeline() lista.")

Función rag_pipeline() lista.


In [57]:
import time
preguntas_prueba = [
    "¿Cuáles son los requisitos para inscribirse en los programas presenciales?",
    "¿Cuales son las condiciones favorables para la admisión de bachilleres?",
    "¿A qué se compromete el estudainte cuando se matricula?",
    "¿Que es la inmovilización?",
    "¿Cuáles son los causales de suspensión o cancelación de la licencia?",
]

print("PRUEBA DEL PIPELINE RAG CON MÚLTIPLES PREGUNTAS")
print("=" * 65)

for pregunta in preguntas_prueba:
    print(f"\n❓ {pregunta}")
    resultado = rag_pipeline(pregunta, 10)
    print(f"🤖 {resultado['respuesta']}")
    print(f"   (~{resultado['tokens_contexto_aprox']} tokens | {len(resultado['fragmentos'])} fragmentos)")
    print("-" * 65)
    time.sleep(10)


PRUEBA DEL PIPELINE RAG CON MÚLTIPLES PREGUNTAS

❓ ¿Cuáles son los requisitos para inscribirse en los programas presenciales?
🤖 **Requisitos para inscribirse en los programas presenciales (pregrado)**  

Según el Reglamento Académico Institucional, los aspirantes a los programas de pregrado – que incluyen los programas presenciales – deben presentar la siguiente documentación y cumplir con los requisitos señalados:

1. **Formulario de inscripción debidamente diligenciado**.  
2. **Certificado de calificaciones del pregrado** (o del nivel de estudio previo correspondiente).  
3. **Copia del documento de identidad vigente**.  
4. **Recibo de pago de los derechos de inscripción** por la suma fijada por el Consejo Superior.  
5. **Requisitos adicionales que establezca el programa**, como entrevistas, exámenes específicos o cartas de recomendación.  

(Fuente: Acuerdo No. 01 de 2025 – Reglamento Académico Institucional, pág. 6)  

---

**Nota:**  
- Para programas de posgrado (especializaci